In [ ]:
CREATE OR ALTER VIEW dbo.vw__CM__CPDash_summary_current
AS
SELECT
    p.[CurrSoldToSalesOfficeCd],
    p.[FiscalYearPeriodDate],
    p.[CurrProdDivCd],
    p.[CurrMatGrpCd],

    
    SUM(COALESCE(TRY_CAST(p.[GrossSalesAmount] AS decimal(38,10)), 0)) AS GrossSales,
    SUM(COALESCE(TRY_CAST(p.[Net Sales]       AS decimal(38,10)), 0)) AS NetSales,
    SUM(COALESCE(TRY_CAST(p.[PPDWriteOff]     AS decimal(38,10)), 0)) AS PPDWriteOff,
    SUM(COALESCE(TRY_CAST(p.[COGS - Manual Adjustments] AS decimal(38,10)), 0)) AS COGSManAdj,
    SUM(COALESCE(TRY_CAST(p.[Over/_Under_ Absorption]   AS decimal(38,10)), 0)) AS Absorption,

    
    SUM(
        COALESCE(TRY_CAST(p.[Outbound Freight]            AS decimal(38,10)), 0)
      + COALESCE(TRY_CAST(p.[Outbound Freight - Tracings] AS decimal(38,10)), 0)
      + COALESCE(TRY_CAST(p.[C Freight]                   AS decimal(38,10)), 0)
    ) AS NetOBFreight

FROM dbo.[CPdash_profitability] p
WHERE p.[CurrSoldToSalesOfficeCd] = 'GL'
  AND p.[FiscalYearPeriodDate] IN ('2025007','2025008','2025009')
GROUP BY
    p.[CurrSoldToSalesOfficeCd],
    p.[FiscalYearPeriodDate],
    p.[CurrProdDivCd],
    p.[CurrMatGrpCd];
GO


In [ ]:
CREATE OR ALTER VIEW dbo.vw__CM__CPDash_current
AS
SELECT
    s.*,

    /* PPDWriteOff% : IF NetSales=0 THEN 0 ELSE PPDWriteOff/GrossSales */
    CASE
        WHEN s.NetSales = 0 THEN 0
        ELSE CAST(s.PPDWriteOff AS decimal(38,10)) / NULLIF(CAST(s.GrossSales AS decimal(38,10)), 0)
    END AS [PPDWriteOff%],

    /* COGSManAdj% : IF NetSales=0 THEN 0 ELSE COGSManAdj/NetSales */
    CASE
        WHEN s.NetSales = 0 THEN 0
        ELSE CAST(s.COGSManAdj AS decimal(38,10)) / NULLIF(CAST(s.NetSales AS decimal(38,10)), 0)
    END AS [COGSManAdj%],

    /* (Over)/UnderAbs% : IF NetSales=0 THEN 0 ELSE Absorption/NetSales */
    CASE
        WHEN s.NetSales = 0 THEN 0
        ELSE CAST(s.Absorption AS decimal(38,10)) / NULLIF(CAST(s.NetSales AS decimal(38,10)), 0)
    END AS [(Over)/UnderAbs%],

    /* OutboundFreight% : NetOBFreight/NetSales */
    CASE
        WHEN s.NetSales = 0 THEN 0
        ELSE CAST(s.NetOBFreight AS decimal(38,10)) / NULLIF(CAST(s.NetSales AS decimal(38,10)), 0)
    END AS [OutboundFreight%]

FROM dbo.vw__CM__CPDash_summary_current s;
GO


In [ ]:
SELECT TOP (20) *
FROM dbo.vw__CM__CPDash_current
ORDER BY FiscalYearPeriodDate DESC;


In [ ]:
SELECT
  COUNT(*) AS RowCnt,
  SUM(GrossSales) AS GrossSalesChk,
  SUM(NetSales) AS NetSalesChk,
  SUM(NetOBFreight) AS NetOBFreightChk
FROM dbo.vw__CM__CPDash_current;
